<a href="https://colab.research.google.com/github/pngy87/-PTDLNC-GOOGLE-COLAB/blob/main/Neural_network_MLP__cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. The California Housing


**Multiple Linear regression**

**Prediction model**

In [ ]:
#load data
import gdown
url = "https://drive.google.com/uc?id=1mi0bB0xF3yfM2cjpeZHnmU52ZKNM1Vl6"
file_path = "california_housing.xlsx"
gdown.download(url, file_path, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1mi0bB0xF3yfM2cjpeZHnmU52ZKNM1Vl6
To: /content/california_housing.xlsx
100%|██████████| 1.69M/1.69M [00:00<00:00, 55.9MB/s]


'california_housing.xlsx'

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error


In [ ]:
df = pd.read_excel(file_path)
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41,6.984127,1.023810,322,2.555556,37.88,-122.23,4.526
1,8.3014,21,6.238137,0.971880,2401,2.109842,37.86,-122.22,3.585
2,7.2574,52,8.288136,1.073446,496,2.802260,37.85,-122.24,3.521
3,5.6431,52,5.817352,1.073059,558,2.547945,37.85,-122.25,3.413
4,3.8462,52,6.281853,1.081081,565,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25,5.045455,1.133333,845,2.560606,39.48,-121.09,0.781
20636,2.5568,18,6.114035,1.315789,356,3.122807,39.49,-121.21,0.771
20637,1.7000,17,5.205543,1.120092,1007,2.325635,39.43,-121.22,0.923
20638,1.8672,18,5.329513,1.171920,741,2.123209,39.43,-121.32,0.847


In [ ]:
# Features & target
X = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']]
y = df['MedHouseVal']

# Split (fit scaler only on train via Pipeline)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=20
)

# MLP model in a pipeline
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32),   # bạn có thể thử (128,64,32) hoặc (100,100)
        activation="relu",
        solver="adam",
        alpha=1e-4,                    # L2 regularization
        learning_rate_init=1e-3,
        max_iter=2000,
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=50,
        random_state=20
    ))
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")


# Training info (optional)
mlp = model.named_steps["mlp"]
print("\nMLP training iterations:", mlp.n_iter_)
print("Final training loss:", mlp.loss_)

Mean Squared Error (MSE): 0.2985
RMSE: 0.5463
MAE: 0.3568

MLP training iterations: 1099
Final training loss: 0.08896505803887281


#Universal Bank

In [ ]:
# Download UniversalBank.csv data file and save to Colab
!wget https://github.com/DanielTrieu/data_analytic/raw/refs/heads/main/data/UniversalBank.csv

--2026-01-19 14:03:04--  https://github.com/DanielTrieu/data_analytic/raw/refs/heads/main/data/UniversalBank.csv
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/DanielTrieu/data_analytic/refs/heads/main/data/UniversalBank.csv [following]
--2026-01-19 14:03:05--  https://raw.githubusercontent.com/DanielTrieu/data_analytic/refs/heads/main/data/UniversalBank.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 205666 (201K) [text/plain]
Saving to: ‘UniversalBank.csv’

UniversalBank.csv   100%[===================>] 200.85K  --.-KB/s    in 0.004s  

2026-01-19 14:03:05 (48.8 MB/s) - ‘UniversalBank

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# ---------- helper: confusion matrix summary (thay cho classificationSummary nếu bạn không có hàm đó) ----------
def classificationSummary(y_true, y_pred, title=""):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn) / (tp + tn + fp + fn)
    print(f"\n{title}")
    print("Confusion Matrix:")
    print(cm)
    print(f"Accuracy: {acc:.4f}")
    print(f"TP={tp}, FP={fp}, FN={fn}, TN={tn}")

In [ ]:
# ---------- Load + basic cleaning ----------
bank_df = pd.read_csv("UniversalBank.csv")

# Drop unnecessary columns
bank_df = bank_df.drop(columns=["ID", "ZIP Code"])

# Replace spaces in column names with underscores
bank_df.columns = [c.replace(" ", "_") for c in bank_df.columns]

# Treat 'Education' as categorical and rename categories
bank_df["Education"] = bank_df["Education"].astype("category")
new_categories = {1: "Undergrad", 2: "Graduate", 3: "Advanced/Professional"}
bank_df["Education"] = bank_df["Education"].cat.rename_categories(new_categories)

# Define target (y) and features (X)
y = bank_df["PersonalLoan"]
X = bank_df.drop(columns=["PersonalLoan"])

# ---------- Train/Validation split ----------
train_X, valid_X, train_y, valid_y = train_test_split(
    X, y, test_size=0.4, random_state=1, stratify=y
)

# ---------- Preprocess: scale numeric, one-hot categorical ----------
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

# ---------- MLP model (Neural Network) ----------
mlp_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=2000,
        early_stopping=True,        # dừng sớm theo validation nội bộ lấy từ TRAIN
        validation_fraction=0.1,
        n_iter_no_change=50,
        random_state=1
    ))
])

# Train
mlp_model.fit(train_X, train_y)

# ---------- Evaluate ----------
# Training
train_pred = mlp_model.predict(train_X)
classificationSummary(train_y, train_pred, title="Training")
print("\nTraining Metrics:")
print(classification_report(train_y, train_pred))

# Validation
valid_pred = mlp_model.predict(valid_X)
classificationSummary(valid_y, valid_pred, title="Validation")
print("\nValidation Metrics:")
print(classification_report(valid_y, valid_pred))

# Optional: training info
mlp = mlp_model.named_steps["mlp"]
print("\nMLP iterations:", mlp.n_iter_)
print("Best internal validation score:", getattr(mlp, "best_validation_score_", None))
print("Final loss:", mlp.loss_)


Training
Confusion Matrix:
[[2712    0]
 [   0  288]]
Accuracy: 1.0000
TP=288, FP=0, FN=0, TN=2712

Training Metrics:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2712
           1       1.00      1.00      1.00       288

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000


Validation
Confusion Matrix:
[[1797   11]
 [  32  160]]
Accuracy: 0.9785
TP=160, FP=11, FN=32, TN=1797

Validation Metrics:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1808
           1       0.94      0.83      0.88       192

    accuracy                           0.98      2000
   macro avg       0.96      0.91      0.93      2000
weighted avg       0.98      0.98      0.98      2000


MLP iterations: 151
Best internal validation score: None
Final loss: 0.0001247892438801206
